# NHLA 2-Side Classification

This notebook trains the dual-face board model. It follows the same workflow as the 1-side notebook:
1. tune hyperparameters on a short validation pass,
2. train the final board model with the best settings,
3. save the final metrics for model comparison.

Only run this after the 1-side training is validated.

In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "26-8-18_75VAL"
CSV_PATH = PROJECT_ROOT / "data" / DATASET_NAME / "nhla_dataset" / "nhla_board_labels.csv"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "models" / f"NHLA_Classification_2Side_{RUN_TIMESTAMP}"
OPTUNA_DIR = PROJECT_ROOT / "logs" / "optuna"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / f"NHLA_Classification_2Side_{RUN_TIMESTAMP}"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics" / f"NHLA_Classification_2Side_{RUN_TIMESTAMP}"

# ------------------------------------------------------------------
# Configuration - change these knobs as needed
# ------------------------------------------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 8
CUDA_DEVICE_INDEX = 0
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
SEED = 33
CHECKPOINT_INTERVAL = 5

# Optuna tuning controls
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 50
TUNE_EPOCHS = 40
STUDY_NAME = f"nhla_2side_optuna_{RUN_TIMESTAMP}"
OPTUNA_DB_PATH = OPTUNA_DIR / "NHLA_Classification.db"
OPTUNA_SEARCH_SPACE = {
    "lr": {"low": 1e-5, "high": 1e-2, "log": True},
    "batch_size": [16, 32, 48, 64],
    "weight_decay": {"low": 1e-6, "high": 1e-2, "log": True},
    "encoder_name": ["resnet18", "resnet34", "resnet50"],
}

# Run naming
CHECKPOINT_NAME = f"NHLA_2Side_{RUN_TIMESTAMP}"
LOG_NAME = f"NHLA_2Side_{RUN_TIMESTAMP}"
METRICS_RUN_NAME = f"nhla_2side_{RUN_TIMESTAMP}"

# Model config
MODEL_NAME = "NHLA_2Side_ResNet18"
ENCODER_NAME = "resnet18"  # options: resnet18, resnet34, resnet50
PRETRAINED = True

print(f"CSV path: {CSV_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"TensorBoard dir: {TENSORBOARD_DIR}")
print(f"Metrics dir: {METRICS_DIR}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Num workers: {NUM_WORKERS}")
print(f"CUDA device index: {CUDA_DEVICE_INDEX}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")
print(f"Checkpoint name: {CHECKPOINT_NAME}")
print(f"Log name: {LOG_NAME}")
print(f"Metrics run name: {METRICS_RUN_NAME}")
print(f"Encoder: {ENCODER_NAME}, pretrained: {PRETRAINED}")
print(f"Optuna enabled: {OPTUNA_ENABLED}, trials: {OPTUNA_TRIALS}")
print("Optuna search space:")
print(json.dumps(OPTUNA_SEARCH_SPACE, indent=2, default=str))

CSV path: /workspaces/Oak-Defect-Detection/data/26-8-18_75VAL/nhla_dataset/nhla_board_labels.csv
Output dir: /workspaces/Oak-Defect-Detection/models/NHLA_Classification_2Side_20260825_153018
TensorBoard dir: /workspaces/Oak-Defect-Detection/logs/tensorboard/NHLA_Classification_2Side_20260825_153018
Metrics dir: /workspaces/Oak-Defect-Detection/data/metrics/NHLA_Classification_2Side_20260825_153018
Image size: 224
Batch size: 16
Num workers: 8
CUDA device index: 0
Checkpoint interval: 5
Checkpoint name: NHLA_2Side_20260825_153018
Log name: NHLA_2Side_20260825_153018
Metrics run name: nhla_2side_20260825_153018
Encoder: resnet18, pretrained: True
Optuna enabled: True, trials: 50
Optuna search space:
{
  "lr": {
    "low": 1e-05,
    "high": 0.01,
    "log": true
  },
  "batch_size": [
    16,
    32,
    48,
    64
  ],
  "weight_decay": {
    "low": 1e-06,
    "high": 0.01,
    "log": true
  },
  "encoder_name": [
    "resnet18",
    "resnet34",
    "resnet50"
  ]
}


In [ ]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import notebooks.NHLA_Classification_2Side as nhla_2side_module
importlib.reload(nhla_2side_module)
train_2side = nhla_2side_module.train_2side

result = train_2side(
    csv_path=str(CSV_PATH),
    output_dir=str(OUTPUT_DIR),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_split=VAL_SPLIT,
    seed=SEED,
    optuna_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
    tune_epochs=TUNE_EPOCHS,
    encoder_name=ENCODER_NAME,
    pretrained=PRETRAINED,
    search_space=OPTUNA_SEARCH_SPACE,
    study_name=STUDY_NAME,
    db_path=str(OPTUNA_DB_PATH),
    checkpoint_name=CHECKPOINT_NAME,
    log_name=LOG_NAME,
    metrics_run_name=METRICS_RUN_NAME,
    tensorboard_dir=str(TENSORBOARD_DIR),
    metrics_dir=str(METRICS_DIR),
    checkpoint_interval=CHECKPOINT_INTERVAL,
    image_size=IMAGE_SIZE,
    num_workers=NUM_WORKERS,
    cuda_device_index=CUDA_DEVICE_INDEX,
)

print(json.dumps({
    "best_params": result["best_params"],
    "metrics": result["result"]["metrics"],
    "metrics_path": result["result"].get("metrics_path"),
}, indent=2))

2026-08-25 15:30:29.424583: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-25 15:30:29.440961: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-25 15:30:29.446445: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[I 2026-08-25 15:30:33,128] A new study created in RDB with name: nhla_2side_optuna_20260825_153018


Using device: cuda:0 (NVIDIA GeForce RTX 3090)


## Notes

- Optuna is run before the final training step.
- The tuned parameters are saved to `best_params.json` in the metrics directory.
- Final training uses the best parameters rather than the initial defaults.
- The 2-side model keeps its board-level CSV and paired-face architecture.